# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [ ]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [ ]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [ ]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [ ]:
def ajouter_features(df):
    """Ajoute une indicatrice d'âge supérieur à 60 ans."""
    df = df.copy()
    if "age" in df.columns:
        df["age_plus_60"] = (df["age"] > 60).astype("int8")
    return df


### La chaîne — une seule, des deux côtés


In [ ]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [ ]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


### La soumission


In [ ]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [ ]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [ ]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "faoadj3-cell"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


### L'interface, telle qu'on l'appelle


In [ ]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [ ]:
part_absents_controle = 100 * yval.mean()
ma_reference = 100 * max(yval.mean(), 1-yval.mean())
pred = np.full(len(yval), int(ytr.mode().iloc[0]), dtype=int)
ecart_a_train = round(ma_reference-reference, 4)
ecart = 83-ma_reference
ecart_en_barres = round(ecart/erreur_type, 4)
reponse_au_collegue = (f"Référence sur le contrôle: {ma_reference:.2f} % (référence train: {reference:.2f} %). "
                       f"83 % la dépasse de {ecart:.2f} point(s), soit {ecart_en_barres:.2f} barre(s); "
                       "en dessous de deux barres, ce n'est pas une différence concluante.")
assert abs((pred==yval).mean()*100-ma_reference) < 1e-10

### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [ ]:
colonnes_a_balayer = [c for c in mon_train.columns if c != CIBLE]
def _balayage(col, filtre=False):
    g = pd.DataFrame({"x": mon_train[col], "y": mon_train[CIBLE]}).groupby("x", dropna=False, observed=True)["y"].agg(["mean", "size"])
    if filtre: g = g[g["size"] >= 500]
    if len(g) < 2: return None
    return {"colonne":col, "ecart_points":100*(g["mean"].max()-g["mean"].min()), "plus_petit_groupe":int(g["size"].min()), "groupes_500":int((g["size"]>=500).sum())}
_rows = [_balayage(c) for c in colonnes_a_balayer]
balayage_brut = pd.DataFrame([r for r in _rows if r is not None]).set_index("colonne").sort_values("ecart_points", ascending=False)
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = float(balayage_brut.iloc[0]["ecart_points"])
groupes_de_500 = int(balayage_brut.iloc[0]["groupes_500"])
_rows_filtres = [_balayage(c, True) for c in colonnes_a_balayer]
classement = pd.DataFrame([r for r in _rows_filtres if r is not None]).set_index("colonne").sort_values("ecart_points", ascending=False)
plus_petit_groupe_du_classement = int(classement.iloc[0]["plus_petit_groupe"])
meilleure_colonne = classement.index[0]
ecart_filtre = float(classement.iloc[0]["ecart_points"])
ce_que_le_critere_ne_protege_pas = f"Le filtre retire les groupes minuscules, mais ne prouve pas qu'une colonne existe au moment de la décision. {meilleure_colonne} peut encore être une fuite ou un proxy; il faut vérifier son sens métier et sa date de disponibilité."
lecture = f"{colonne_piegee} gagne sans filtre avec {ecart_non_filtre:.2f} points parce que ses petits groupes portent des taux extrêmes. Après le seuil de 500 lignes, {meilleure_colonne} arrive en tête avec {ecart_filtre:.2f} points: la première victoire était un artefact d'effectif."
display(balayage_brut.head(10)); display(classement.head(10))

### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [ ]:
constat("aucune ligne strictement dupliquée",
        f"{int(train.duplicated().sum())} ligne(s) en doublon",
        f"sur {len(train)} lignes, soit {100*train.duplicated().mean():.3f} %",
        "garder les lignes: aucun doublon exact à retirer")
constat("l'âge varie selon les patients",
        f"{train['age'].nunique()} âges distincts",
        f"contre 1 valeur si l'âge était constant, sur {len(train)} rendez-vous",
        "garder l'âge comme mesure; mesurer si un seuil ajoute de l'information")

### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [ ]:
pred_age = 0.0
pred_doublons = 0.0
pred_taille = 0.0
X_age_corrige = X.copy()
for col in ("age", "age_mois"):
    if col in X_age_corrige:
        X_age_corrige.loc[X_age_corrige[col] < 0, col] = np.nan
        X_age_corrige[col] = X_age_corrige[col].fillna(X_age_corrige[col].median())
a,b,c,d = train_test_split(X_age_corrige,y,test_size=0.2,stratify=y,random_state=0)
m_age = HistGradientBoostingClassifier(random_state=0).fit(a,c)
s_avec_age = 100*m_age.score(b,d)
X_sans_age = X_age_corrige.drop(columns=[c for c in ("age","age_mois") if c in X_age_corrige])
a,b,c,d = train_test_split(X_sans_age,y,test_size=0.2,stratify=y,random_state=0)
m_sans_age = HistGradientBoostingClassifier(random_state=0).fit(a,c)
s_sans_age = 100*m_sans_age.score(b,d)
age_en_barres = (s_avec_age-s_sans_age)/erreur_type
Xlin = pd.get_dummies(X_age_corrige,columns=CAT).fillna(0)
Xlin_sans = Xlin.drop(columns=[c for c in ("age","age_mois") if c in Xlin])
a,b,c,d=train_test_split(Xlin,y,test_size=0.2,stratify=y,random_state=0)
lin_m_age=LogisticRegression(solver="newton-cholesky",max_iter=1000).fit(a,c)
lin_avec_age=100*lin_m_age.score(b,d)
a,b,c,d=train_test_split(Xlin_sans,y,test_size=0.2,stratify=y,random_state=0)
lin_m_sans=LogisticRegression(solver="newton-cholesky",max_iter=1000).fit(a,c)
lin_sans_age=100*lin_m_sans.score(b,d)
lin_age_en_barres=(lin_avec_age-lin_sans_age)/erreur_type
verdict_age=(f"La correction de l'âge change l'exactitude des arbres de {s_avec_age-s_sans_age:.2f} point(s) ({age_en_barres:.2f} barre(s)) et celle du linéaire de {lin_avec_age-lin_sans_age:.2f} point(s) ({lin_age_en_barres:.2f} barre(s)). Un arbre peut isoler -1 dans une branche; une somme pondérée l'interprète comme une valeur numérique. La correction est sémantiquement nécessaire même si le gain est faible.")
verdict_doublons=f"Le contrôle trouve {int(train.duplicated().sum())} doublon(s) exact(s): zéro signifie qu'aucune ligne n'est à retirer selon ce contrôle, pas que le test n'a rien appris."

### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [ ]:
def scores_taille(n, seeds):
    n=min(n,len(Xval)); out=[]
    for seed in seeds:
        idx=Xval.sample(n=n,random_state=seed).index
        out.append(100*modele.score(Xval.loc[idx],yval.loc[idx]))
    return out
ecarts_1000=scores_taille(1000,range(20))
ecarts_5000=scores_taille(5000,range(20))
ecarts_1000=[round(v,4) for v in ecarts_1000]; ecarts_5000=[round(v,4) for v in ecarts_5000]
ecart_type_1000=float(np.std(ecarts_1000,ddof=1)); ecart_type_5000=float(np.std(ecarts_5000,ddof=1))
p=score_interne/100
formule_1000=100*(p*(1-p)/min(1000,len(Xval)))**0.5
formule_5000=100*(p*(1-p)/min(5000,len(Xval)))**0.5
rapport_1000=ecart_type_1000/formule_1000; rapport_5000=ecart_type_5000/formule_5000
lecture=f"Quand n est multiplié par 5, la barre théorique passe de {formule_1000:.3f} à {formule_5000:.3f} point(s), soit une division par racine de 5. Les rapports observé/formule sont {rapport_1000:.2f} et {rapport_5000:.2f}; vingt tirages donnent une estimation encore bruyante de l'écart-type."

### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [ ]:
ecarts_25000=scores_taille(25000,range(100,120))
ecart_type_25000=float(np.std(ecarts_25000,ddof=1))
n=min(25000,len(Xval))
formule_25000=100*(p*(1-p)/n)**0.5
rapport_25000=ecart_type_25000/formule_25000
correction_population=(max(0,(len(Xval)-n)/(len(Xval)-1)))**0.5 if len(Xval)>1 else 0
rapport_corrige_population=ecart_type_25000/(formule_25000*correction_population) if correction_population else np.nan
pourquoi_la_formule_deborde=(f"La formule suppose des observations indépendantes; ici, 25 000 lignes représentent {100*n/len(Xval):.1f} % du jeu de contrôle et le tirage est sans remise, donc les observations deviennent dépendantes. Le rapport observé/théorique vaut {rapport_25000:.2f}, et {rapport_corrige_population:.2f} après correction de population finie.")

### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [ ]:
pred_pas="Je m'attends à ce que 0,05 apprenne plus lentement et construise davantage d'arbres, 0,5 soit plus rapide, et 2 s'arrête tôt ou soit moins stable; l'arrêt anticipé peut rapprocher les scores."
reglages=[]
for taux in (0.05,0.5,2.0):
    m=HistGradientBoostingClassifier(learning_rate=taux,random_state=0).fit(Xtr,ytr)
    score=100*m.score(Xval,yval)
    reglages.append({"learning_rate":taux,"score":score,"arbres":m.n_iter_,"en_barres":(score-reference)/erreur_type})
score_separe=("Oui." if max(r['score'] for r in reglages)-min(r['score'] for r in reglages)>2*erreur_type else "Non: le plus grand écart entre les trois scores reste inférieur à deux barres.")
score_separe += " Les valeurs mesurées sont " + str([round(r['score'],2) for r in reglages]) + "."
ce_qui_borne=f"Les nombres d'arbres construits sont {[r['arbres'] for r in reglages]}. L'arrêt anticipé borne le modèle; les scores et n_iter_ permettent de juger si cela confirme la prédiction sur le pas d'apprentissage."
display(pd.DataFrame(reglages))

### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [ ]:
prevoir("retirer zone_clinique","score inchangé","la colonne est constante et ne sépare aucun rendez-vous")
m_donnee=HistGradientBoostingClassifier(random_state=0).fit(Xtr.drop(columns=["zone_clinique"],errors="ignore"),ytr)
s_donnee=essai("retirer zone_clinique",lambda:m_donnee.score(Xval.drop(columns=["zone_clinique"],errors="ignore"),yval),"retrait de la colonne constante","aucun changement attendu")
prevoir("learning_rate 0.05","score proche","le pas change la vitesse d'apprentissage, pas l'information des données")
m_reglage=HistGradientBoostingClassifier(learning_rate=0.05,random_state=0).fit(Xtr,ytr)
s_reglage=essai("learning_rate 0.05",lambda:m_reglage.score(Xval,yval),"learning_rate=0.05","l'arrêt anticipé peut limiter l'effet")
prevoir("régression logistique","score légèrement inférieur","une frontière linéaire capture moins d'interactions")
s_famille=essai("régression logistique",lambda:score_lineaire/100,"LogisticRegression","les arbres captent des seuils et interactions")
natures={"retirer zone_clinique":"donnée","learning_rate 0.05":"réglage","régression logistique":"famille de modèle"}
mesures={"retirer zone_clinique":s_donnee,"learning_rate 0.05":s_reglage,"régression logistique":s_famille}
mes_essais=pd.DataFrame([{"essai":n,"nature":natures[n],"attendu":next(p["attendu"] for p in PREDICTIONS if p["essai"]==n),"mesuré":s,"gain_en_barres":(s-score_interne)/erreur_type,"verdict":"réel" if abs(s-score_interne)>=2*erreur_type else "dans le bruit"} for n,s in mesures.items()])
meilleur=mes_essais.sort_values("mesuré",ascending=False).iloc[0]
optimisme=f"Le meilleur essai est choisi après comparaison sur la même validation; cette sélection surestime probablement son gain. Je m'attends à un gain plus faible pour {meilleur['essai']} au second découpage."
a2,b2,c2,d2=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
if meilleur["essai"]=="retirer zone_clinique":
    m2=HistGradientBoostingClassifier(random_state=0).fit(a2.drop(columns=["zone_clinique"],errors="ignore"),c2); score2=100*m2.score(b2.drop(columns=["zone_clinique"],errors="ignore"),d2)
elif meilleur["essai"]=="learning_rate 0.05":
    m2=HistGradientBoostingClassifier(learning_rate=0.05,random_state=0).fit(a2,c2); score2=100*m2.score(b2,d2)
else:
    score2=score_lineaire
verifie_par_second_tirage=(score2-score_interne)/erreur_type
display(mes_essais)

### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [ ]:
rapport=(score_interne-score_lineaire)/erreur_type
coef=pd.Series(modele_lineaire.coef_[0],index=X_num.columns)
candidats=[c for c in X_num.columns if c.lower().startswith(("quartier_","assurance_"))]
if len(candidats)<3:
    candidats += [c for c in X_num.columns if c not in candidats and c not in ("age","age_mois","montant_acte","cout_acte") and not c.lower().startswith(("dossier","patient"))]
trois_coefficients={c:float(coef[c]) for c in candidats[:3]}
pourquoi_pas_les_dupliquees="Les colonnes dupliquées portent la même information sur une autre échelle; leurs coefficients se partagent l'effet et deviennent instables, donc ils ne se lisent pas séparément."
quartier_provoque="Un coefficient de quartier décrit une association conditionnelle, pas une cause: la composition des patients ou l'accès aux soins peut expliquer l'écart observé."
cout_et_rapport=f"Le modèle linéaire demande le codage des catégories et le traitement des valeurs manquantes, mais fournit un coefficient signé par colonne. Son écart au modèle d'arbres vaut {rapport:.2f} barre(s); sous deux barres, je ne conclus pas à une différence."

### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [ ]:
constat_d_origine="l'âge varie selon les patients"
prevoir("âge supérieur à 60 ans","gain faible","l'âge brut contient déjà cette information")
n_colonnes_avant=X.shape[1]
score_avant_feature=score_interne
score_apres_feature=remesurer()
noter("âge supérieur à 60 ans", "ajout de age_plus_60", "gain attendu faible: l'âge brut contient déjà cette information", score_apres_feature)
gain_de_ma_feature=(score_apres_feature-score_avant_feature)/erreur_type
Xval_correct=ajouter_features(Xval)
# Contournement: la transformation est oubliée, la nouvelle indicatrice est fixée à zéro.
Xval_contourne=Xval.copy()
Xval_contourne["age_plus_60"]=0
score_contourne=100*modele.score(Xval_contourne,yval)
ecart_du_contournement=(score_contourne-score_apres_feature)/erreur_type
quelle_etape_oubliee="La création de age_plus_60 a été oubliée lors de la préparation de la validation: le modèle reçoit une valeur neutre au lieu de la feature calculée, ce qui rend entraînement et prédiction incohérents."

### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [ ]:
colonnes_communes=[c for c in train.columns if c in test.columns and c not in (ID,CIBLE)]
ecarts=pd.DataFrame({"remplissage_train":[100*train[c].notna().mean() for c in colonnes_communes],"remplissage_test":[100*test[c].notna().mean() for c in colonnes_communes]},index=colonnes_communes)
ecarts["ecart_abs"]=(ecarts["remplissage_train"]-ecarts["remplissage_test"]).abs()
ecarts=ecarts.sort_values("ecart_abs",ascending=False)
colonne_trouvee=ecarts.index[0]
ecart_remplissage=float(ecarts.iloc[0]["ecart_abs"])
remplissage_du_test=float(100*test[colonne_trouvee].notna().mean())
colonne_derivee=ecarts.index[1]
pourquoi_pas_la_derivee=f"La deuxième colonne, {colonne_derivee}, a un écart de {ecarts.iloc[1]['ecart_abs']:.2f} points, mais il s'agit d'une dérive de remplissage et non d'une disparition complète."
Xq=X.loc[mon_train.index].copy(); yq=mon_train[CIBLE]
if colonne_trouvee in train.columns and colonne_trouvee not in Xq.columns:
    vals=train.loc[mon_train.index,colonne_trouvee]
    if not pd.api.types.is_numeric_dtype(vals): vals=vals.fillna("").astype("category")
    Xq[colonne_trouvee]=vals
qa,qb,qy,qv=train_test_split(Xq,yq,test_size=0.2,stratify=yq,random_state=0)
m_avec=HistGradientBoostingClassifier(random_state=0).fit(qa,qy)
score_avec=100*m_avec.score(qb,qv)
m_sans=HistGradientBoostingClassifier(random_state=0).fit(qa.drop(columns=[colonne_trouvee],errors="ignore"),qy)
score_sans=100*m_sans.score(qb.drop(columns=[colonne_trouvee],errors="ignore"),qv)
gain_interne=score_avec-score_sans; gain_en_barres=gain_interne/erreur_type
qb_sim=qb.copy(); n_manquants=int(round((1-remplissage_du_test/100)*len(qb_sim)))
idx_manquants=np.random.RandomState(0).choice(qb_sim.index.to_numpy(),size=n_manquants,replace=False) if n_manquants else []
if pd.api.types.is_categorical_dtype(qb_sim[colonne_trouvee]):
    qb_sim[colonne_trouvee]=qb_sim[colonne_trouvee].cat.add_categories(["__MANQUANT__"])
    qb_sim.loc[idx_manquants,colonne_trouvee]="__MANQUANT__"
else: qb_sim.loc[idx_manquants,colonne_trouvee]=np.nan
score_simule=100*m_avec.score(qb_sim,qv)
chute_en_barres=(score_simule-score_avec)/erreur_type
decision_A_ECARTER=f"Écarter {colonne_trouvee} est prudent car {remplissage_du_test:.1f} % seulement du test porte une valeur. En interne, le gain vaut {gain_interne:.2f} point(s), soit {gain_en_barres:.2f} barre(s); après simulation du taux de trous du test, le score est {score_simule:.2f} % ({chute_en_barres:.2f} barre(s) par rapport au modèle avec la colonne). L'écarter est justifié par sa disponibilité en production, même si son gain interne est réel."
display(ecarts.head())

### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [ ]:
def suspects(df,cible):
    y_local=df[cible]; lignes=[]
    identifiants={ID,cible,"dossier_id","patient_id"}
    for col in df.columns:
        if col in identifiants or col.lower().endswith("_id"): continue
        groupes=[df.loc[y_local==k,col].dropna() for k in (0,1)]
        remplis=[float(g.size/ max(1,(y_local==k).sum())*100) for k,g in zip((0,1),groupes)]
        ecart=remplis[1]-remplis[0]
        if abs(ecart)>50:
            lignes.append({"colonne":col,"ecart_remplissage_points":ecart,"signe":"vide","valeur":"taux de remplissage très différent selon la cible"})
            continue
        v0=set(groupes[0].astype(str)); v1=set(groupes[1].astype(str)); exclusifs=v0.symmetric_difference(v1)
        if exclusifs and remplis[0]>0 and remplis[1]>0:
            lignes.append({"colonne":col,"ecart_remplissage_points":ecart,"signe":"valeur","valeur":", ".join(list(exclusifs)[:5])})
    return pd.DataFrame(lignes,columns=["colonne","ecart_remplissage_points","signe","valeur"])
table_suspects=suspects(train,CIBLE)
classements=[]
for _,r in table_suspects.iterrows():
    nom=r["colonne"]
    if nom in ("motif_absence","duree_consultation","constantes_relevees"):
        verdict="fuite"; justification=f"{nom} décrit le résultat du rendez-vous ou un manque structurel qui suit l'issue; cette information ne serait pas disponible au moment de décider."
    elif nom in ("age","age_mois","taille_cm"):
        verdict="pas une fuite"; justification=f"{nom} décrit une mesure du patient disponible avant le rendez-vous; une valeur impossible appelle un contrôle de saisie, mais ne révèle pas en soi l'issue."
    else:
        verdict="à discuter"; justification=f"Pour {nom}, le signe statistique ne suffit pas: il faut vérifier ce que signifie la valeur et quand elle est renseignée avant de conclure à une fuite."
    classements.append({"colonne":nom,"verdict":verdict,"justification":justification})
classement_suspects=pd.DataFrame(classements,columns=["colonne","verdict","justification"])
sentinelle_trouvee="age = -1 (valeur impossible)"
masque_sentinelle=train["age"]==-1
lignes_porteuses=int(masque_sentinelle.sum())
taux_sur_ces_lignes=100*train.loc[masque_sentinelle,CIBLE].mean() if lignes_porteuses else 0.0
verdict_sentinelle=f"Défaut de saisie: -1 n'est pas un âge possible et les lignes concernées sont peu nombreuses ({lignes_porteuses}); c'est une valeur manquante codée, pas une information apparue après l'issue. Le taux d'absence sur ces lignes est {taux_sur_ces_lignes:.2f} %, contre {100*train[CIBLE].mean():.2f} % au global; toute association éventuelle doit être vérifiée avant correction."
display(table_suspects); display(classement_suspects)

### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [ ]:
annonce="À délai égal, je m'attends à un effet faible ou nul du rappel; l'écart global peut venir du délai de rendez-vous. Je changerais d'avis si plusieurs tranches assez grandes montraient un écart supérieur à deux barres dans le même sens."
col_rappel=next(c for c in train.columns if "rappel" in c.lower())
col_delai=next(c for c in train.columns if "delai" in c.lower() or "délai" in c.lower())
df_r=mon_train[[col_rappel,col_delai,CIBLE]].copy()
df_r["avec"]=df_r[col_rappel].astype(str).str.lower().isin(["1","oui","true","avec","sms","rappel"])
taux_avec=100*df_r.loc[df_r.avec,CIBLE].mean(); taux_sans=100*df_r.loc[~df_r.avec,CIBLE].mean()
ecart_global=taux_avec-taux_sans
delai_median_avec=float(df_r.loc[df_r.avec,col_delai].median()); delai_median_sans=float(df_r.loc[~df_r.avec,col_delai].median())
df_r["tranche"]=pd.qcut(df_r[col_delai],q=4,duplicates="drop")
tranches=[]; tranches_exclues=[]
for nom,g in df_r.groupby("tranche",observed=True,dropna=False):
    a=g[g.avec]; s=g[~g.avec]; n=len(g)
    if n<500 or len(a)==0 or len(s)==0:
        tranches_exclues.append({"tranche":str(nom),"lignes":n,"raison":"moins de 500 lignes ou l'un des groupes absent; comparaison trop instable"}); continue
    p1=a[CIBLE].mean(); p0=s[CIBLE].mean(); ec=100*(p1-p0)
    barre=100*(p1*(1-p1)/len(a)+p0*(1-p0)/len(s))**0.5
    tranches.append({"tranche":str(nom),"taux_avec":100*p1,"taux_sans":100*p0,"n_avec":len(a),"n_sans":len(s),"lignes":n,"ecart":ec,"barre":barre,"en_barres":ec/barre if barre else np.nan})
variable_qui_explique=f"Le délai de rendez-vous: médiane {delai_median_avec:.1f} jours avec rappel et {delai_median_sans:.1f} jours sans rappel. Une composition différente par délai peut expliquer l'écart global; les taux par tranche vérifient s'il subsiste un écart à délai égal."
experience_qui_trancherait="Tirer au hasard, parmi des rendez-vous comparables, ceux qui reçoivent un rappel et ceux qui n'en reçoivent pas.
Comparer les taux d'absence à délai égal, avec un effectif suffisant et la même définition de l'issue.
La randomisation rend les groupes comparables en moyenne et permet d'attribuer plus honnêtement une différence au rappel."
display(pd.DataFrame(tranches)); display(pd.DataFrame(tranches_exclues))

### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [ ]:
hypothese("âge supérieur à 60 ans","L'indicateur age_plus_60 pourrait augmenter le taux d'absence; signe attendu positif.","Ajouter age_plus_60 puis comparer les exactitudes sur le même découpage et mesurer le gain en barres.","Contredite si le gain est inférieur à deux barres ou si le taux d'absence des plus de 60 ans n'est pas supérieur.")
hypothese("délai long","Un délai plus long pourrait augmenter l'absence; signe attendu positif.","Comparer le modèle de base à l'ajout d'un indicateur de délai au-dessus de la médiane, sur le même découpage.","Contredite si le gain reste sous deux barres et si la tranche longue n'a pas un taux d'absence supérieur.")
hypothese("rappel à délai égal","À délai égal, le rappel devrait réduire légèrement l'absence; signe attendu négatif.","Comparer les taux avec/sans rappel par tranches d'au moins 500 lignes et exprimer l'écart en barres.","Contredite si les tranches assez grandes montrent un écart positif supérieur à deux barres de façon cohérente.")
prevoir("âge supérieur à 60 ans","gain faible, sous deux barres","l'âge brut porte déjà cette information et un seuil ne devrait pas ajouter beaucoup")
score_hypothese=essai("âge supérieur à 60 ans",lambda:modele.score(ajouter_features(Xval),yval),"ajout de age_plus_60","gain attendu faible car le seuil est déjà accessible au modèle")
gain_hypothese=score_hypothese-score_interne
confirmee_ou_contredite="confirmée" if gain_hypothese>=2*erreur_type else "contredite"
suite_donnee=f"Le gain vaut {gain_hypothese:.2f} point(s), soit {gain_hypothese/erreur_type:.2f} barre(s); le critère annoncé était deux barres. Je garde la feature seulement si ce seuil est dépassé, sinon je documente le résultat et conserve le modèle plus simple."

### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [ ]:
familles={"id_client":"identifiant","date_defaut":"connue après l'issue","statut_6_mois":"connue après l'issue","montant":"utilisable","canal":"risque de disparition","score_interne":"utilisable"}
justifications={
"id_client":"Cet identifiant désigne le client et ne décrit pas le risque de sa demande; il ne doit pas apprendre au modèle.",
"date_defaut":"La date n'existe que si le défaut survient et se situe après la décision de crédit; elle révèle l'issue.",
"statut_6_mois":"Ce statut est relevé six mois après l'octroi et n'est donc pas disponible au moment où le modèle décide.",
"montant":"Le montant demandé est connu au moment de la décision et décrit directement la demande de crédit.",
"canal":"Le canal peut être connu au départ, mais son alimentation peut changer ou disparaître en production; sa disponibilité doit être contrôlée.",
"score_interne":"Ce score est calculé au moment de la demande et constitue une mesure de risque disponible avant la décision."}

## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [ ]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
